# Hàm load dữ liệu

In [1]:
# import số thư viện cần thiết
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, concatenate, Reshape, Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf


In [2]:
model_multimodal = load_model('Multimodal_3component.h5', compile=False)

In [3]:
import pandas as pd
import numpy as np
from numpy import savetxt, loadtxt

def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")
    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels



In [4]:
X_train_opcode_8, X_test_opcode_8, X_train_source_8, X_test_source_8, y_train_8, y_test_8, labels_dataset_full = load_dataset(8)

Load Features codeBERT success!!
Load Data for 8-Label MultiLabel success!!


In [5]:
# Load GNN
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np

# Load data
input_file = 'DATASET/gnn_input_binary_labels.pkl'

with open(input_file, 'rb') as f:
    data = pickle.load(f)

loaded_feature_matrices = data['features']
adj_matrices = data['adj_matrices']
# loaded_all_labels = data['labels']
# Define the target dimension
target_dim = 6000

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim))
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features

# Function to create PyTorch Geometric Data object from feature matrices and labels
def create_pyg_data(features, adj_matrix, labels, target_dim):
    num_nodes = features.shape[0]
    edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset_full)]

# Create DataLoader
train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


In [6]:
from torch_geometric.nn import GATConv
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=0.3)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()

        x = global_mean_pool(x, batch)  # Pooling layer

        x = F.dropout(x, p=0.3, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification

        return x

# Load mô hình GNN
model_path = 'Unimodal/GAT_30E.pth'

# Adjust the in_channels, hidden_channels, and out_channels parameters based on the training settings
gnn_model = GAT(in_channels=6000, hidden_channels=64, out_channels=8)
gnn_model.load_state_dict(torch.load(model_path))
gnn_model.eval()  # Set the model to evaluation mode

# Function to get GNN outputs for a given DataLoader
def get_gnn_outputs(loader):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = gnn_model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

# Get GNN outputs for training and testing data
gnn_train_outputs = get_gnn_outputs(train_loader)
gnn_test_outputs = get_gnn_outputs(test_loader)

C:\Users\Admin\AppData\Local\Temp\ipykernel_18892\214182964.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gnn_model.load_state_dict(torch.load(model_path))


In [7]:
bert_model = load_model("Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5", compile=False)
lstm_model = load_model("Unimodal/CodeBERT_30E.h5", compile=False)
# Set the models to non-trainable
bert_model.trainable = False
lstm_model.trainable = False

# Combine the outputs of both models
bert_output = bert_model.output
lstm_output = lstm_model.output
gnn_output = tf.keras.layers.Input(shape=(8,), name="gnn_output")  # Kích thước đầu ra của GNN

# Tắt trainable cho GNN
gnn_output._keras_history[0].trainable = False
concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)

# Feature extractor part
concatenated_reshaped = Reshape((-1, 1))(concatenated)  # Adjust to match Conv1D input
conv_out = Conv1D(64, 3, activation='relu')(concatenated_reshaped)
flatten_out = Flatten()(conv_out)
dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
dense_features_1 = Dropout(0.3, name='drop_multilabel')(dense_features)
dense_features_2 = Dense(64, activation='relu', name='dense_multilabel_2')(dense_features_1)
dense_features_3 = Dropout(0.3, name='drop_multilabel_2')(dense_features_2)
# Output for the initial_label_count task
output_labels = Dense(8, activation='sigmoid', name='output_labels')(dense_features_3)

# Create initial model
model = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=output_labels)

# Compile the model
METRICS = [
tf.keras.metrics.BinaryAccuracy(name='accuracy'),
tf.keras.metrics.Precision(name='precision'),
tf.keras.metrics.Recall(name='recall')
]

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS, run_eagerly=True)

In [8]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 280)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 280, 286)     9724000     ['input_1[0][0]']                
                                                                                                  
 sourcecode_features (InputLaye  [(None, 768)]       0           []                               
 r)                                                                                               
                                                                                                  
 bidirectional (Bidirectional)  (None, 256)          424960      ['embedding[0][0]']          

In [9]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)

In [ ]:
model.fit([X_train_opcode_8, X_train_source_8,gnn_train_outputs], y_train_8, epochs=30, batch_size=32, callbacks=[early_stopping])

Epoch 1/30
992/992 [==============================] - 2199s 2s/step - loss: 0.3017 - accuracy: 0.8768 - precision: 0.8350 - recall: 0.7857
Epoch 2/30
862/992 [=========================>....] - ETA: 4:56 - loss: 0.2670 - accuracy: 0.8934 - precision: 0.8557 - recall: 0.8176

# Test

In [ ]:
def apply_threshold(y_pred_prob, threshold=0.5):
    y_pred_prob = np.array(y_pred_prob)
    y_pred_thresh = np.zeros_like(y_pred_prob)
    y_pred_thresh[y_pred_prob >= threshold] = 1
    return y_pred_threshƯ

In [ ]:
y_pred_prob = model.predict([X_test_opcode_8, X_test_source_8, gnn_test_outputs])

248/248 [==============================] - 9s 37ms/step


In [ ]:
y_pred = apply_threshold(y_pred_prob, threshold=0.5)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

for i, label in enumerate(label_names):
    precision = precision_score(y_test_8[:, i], y_pred[:, i])
    recall = recall_score(y_test_8[:, i], y_pred[:, i])
    f1 = f1_score(y_test_8[:, i], y_pred[:, i])
    accuracy = accuracy_score(y_test_8[:, i], y_pred[:, i])
    # Tính confusion matrix cho nhãn i
    tn, fp, fn, tp = confusion_matrix(y_test_8[:, i], y_pred[:, i]).ravel()

    # Tính FPR và FNR32
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    print(f"Metrics for {label}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  FPR:       {fpr:.4f}")
    print(f"  FNR:       {fnr:.4f}")
    print()

Metrics for Other:
  Accuracy:  0.8820
  Precision: 0.9013
  Recall:    0.8885
  F1-Score:  0.8949
  FPR:       0.1265
  FNR:       0.1115

Metrics for access_control:
  Accuracy:  0.9581
  Precision: 0.9248
  Recall:    0.5499
  F1-Score:  0.6897
  FPR:       0.0041
  FNR:       0.4501

Metrics for arithmetic:
  Accuracy:  0.9477
  Precision: 0.9654
  Recall:    0.9659
  F1-Score:  0.9656
  FPR:       0.1102
  FNR:       0.0341

Metrics for denial_service:
  Accuracy:  0.9639
  Precision: 0.8906
  Recall:    0.9023
  F1-Score:  0.8964
  FPR:       0.0232
  FNR:       0.0977

Metrics for front_running:
  Accuracy:  0.9541
  Precision: 0.8728
  Recall:    0.8073
  F1-Score:  0.8388
  FPR:       0.0204
  FNR:       0.1927

Metrics for reentrancy:
  Accuracy:  0.9073
  Precision: 0.8824
  Recall:    0.8022
  F1-Score:  0.8404
  FPR:       0.0468
  FNR:       0.1978

Metrics for time_manipulation:
  Accuracy:  0.9589
  Precision: 0.8971
  Recall:    0.4200
  F1-Score:  0.5722
  FPR:       

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

subset_acc = accuracy_score(y_test_8, y_pred)

print("Subset Accuracy:", round(subset_acc, 4))
print("\nClassification Report:")
print(classification_report(
    y_test_8,
    y_pred,
    target_names=label_names
))